In [1]:
# uv venv .venv
# source .venv/bin/activate
# uv add pandas numpy scikit-learn jupyterlab
# uv sync
# uv run app.py


## after cloning
# uv sync

# .gitignore
## .venv/
## __pycache__/
## .ipynb_checkpoints/

## Creating a react agent(with tools, RuntimeContext and a SystemPrompt)

In [1]:
from langchain.agents import create_agent

In [2]:
from __future__ import annotations
import os
from dataclasses import dataclass
from pathlib import Path
from langchain.agents import create_agent
from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

EXAMPLE_QUESTION = "Which table has the largest number of entries?"
SYSTEM_PROMPT = """You are a careful SQLite analyst.
Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Do NOT ask the user for permission before calling tools.
- If data is needed, execute the query immediately.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

@dataclass
class RuntimeContext:
    db: SQLDatabase

def load_config():
    output_config = {
        "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
        "LANGSMITH_API_KEY": os.getenv("LANGSMITH_API_KEY"),
        "LANGSMITH_TRACING_V2": os.getenv("LANGSMITH_TRACING_V2"),
        "LANGSMITH_ENDPOINT": os.getenv("LANGSMITH_ENDPOINT"),
        "MODEL_NAME": "openai:gpt-5-mini",
        "PROJECT_NAME": os.getenv("LANGSMITH_PROJECT"),
        "DB_PATH": "Chinook.db",
    }
    return output_config

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite query and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    try:
        return db.run(query)
    except Exception as exc:
        return f"Error: {exc}"

script_dir = Path.cwd()
config = load_config()
db_path = Path(config["DB_PATH"])
db = SQLDatabase.from_uri(f"sqlite:///{db_path}")

agent = create_agent(
    model=config["MODEL_NAME"],
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

for step in agent.stream(
    {"messages": EXAMPLE_QUESTION},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    messages = step.get("messages", [])
    if messages:
        messages[-1].pretty_print()


================================ Human Message =================================

Which table has the largest number of entries?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_GiipMRT8sTSZ5JMHGFLU6rvX)
 Call ID: call_GiipMRT8sTSZ5JMHGFLU6rvX
  Args:
    query: SELECT m.name AS table_name, COALESCE(s.stat,'') AS stat_text
FROM sqlite_master m
LEFT JOIN sqlite_stat1 s ON s.tbl = m.name
WHERE m.type='table' AND m.name NOT LIKE 'sqlite_%'
ORDER BY CASE WHEN s.stat GLOB '*[0-9]*' THEN CAST(s.stat AS INTEGER) ELSE 0 END DESC
LIMIT 1;
================================= Tool Message =================================
Name: execute_sql

Error: (sqlite3.OperationalError) no such table: sqlite_stat1
[SQL: SELECT m.name AS table_name, COALESCE(s.stat,'') AS stat_text
FROM sqlite_master m
LEFT JOIN sqlite_stat1 s ON s.tbl = m.name
WHERE m.type='table' AND m.name NOT LIKE 'sqlite_%'
ORDER BY CASE WHEN s.stat GLOB '*[0-9]*' THEN CAST(s.s

## Exploring the message options in langchain

In [3]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
    ToolMessage,
    convert_to_messages,
    messages_to_dict,
)

# 1) Build message objects for a real question against Chinook.db
conversation_input = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="Which table has the largest number of entries?"),
]


# 2) Let your existing agent produce AI + tool messages dynamically
result = agent.invoke(
    {"messages": conversation_input},
    context=RuntimeContext(db=db),
)
messages = result["messages"]
print("Message types returned by the agent:")
print([m.type for m in messages]) # 'system', 'human', 'ai', 'tool', 'ai', 'tool', 'ai'


# 3) Inspect dynamic AI/tool messages (no hardcoded AIMessage)
last_ai = next((m for m in reversed(messages) if isinstance(m, AIMessage)), None)
tool_msgs = [m for m in messages if isinstance(m, ToolMessage)]


# 4) Convert role-style payloads to Message objects
raw_payload = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "hello"},
    {"role": "assistant", "content": "hi"},
]
converted = convert_to_messages(raw_payload) # dictionaries to proper message objects
print("\nConverted roles -> message types:", [m.type for m in converted])

# 5) Serialize messages for storage/logging
serialized = messages_to_dict(messages) # list of dictionries
print("Serialized message count:", len(serialized))

# 6) RemoveMessage example using a real message id from this run
remove_last = RemoveMessage(id=messages[-1].id)
print("\nRemoveMessage example:", remove_last)


Message types returned by the agent:
['system', 'human', 'ai', 'tool', 'ai', 'tool', 'ai']

Converted roles -> message types: ['system', 'human', 'ai']
Serialized message count: 7

RemoveMessage example: content='' additional_kwargs={} response_metadata={} id='lc_run--019e540e-7274-7ac3-af65-621d86033351-0'


In [4]:
messages

[SystemMessage(content="You are a careful SQLite analyst.\nRules:\n- Think step-by-step.\n- When you need data, call the tool `execute_sql` with ONE SELECT query.\n- Do NOT ask the user for permission before calling tools.\n- If data is needed, execute the query immediately.\n- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.\n- Limit to 5 rows unless the user explicitly asks otherwise.\n- If the tool returns 'Error:', revise the SQL and try again.\n- Prefer explicit column lists; avoid SELECT *.\n", additional_kwargs={}, response_metadata={}, id='8102d112-9856-44a7-b133-e4f97b186d6d'),
 HumanMessage(content='Which table has the largest number of entries?', additional_kwargs={}, response_metadata={}, id='a8e79bde-19e9-44f0-81a2-4e25d00c9aaf'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 748, 'prompt_tokens': 378, 'total_tokens': 1126, 'completion_tokens_details': {'accepted_prediction_tok

In [5]:
serialized

[{'type': 'system',
  'data': {'content': "You are a careful SQLite analyst.\nRules:\n- Think step-by-step.\n- When you need data, call the tool `execute_sql` with ONE SELECT query.\n- Do NOT ask the user for permission before calling tools.\n- If data is needed, execute the query immediately.\n- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.\n- Limit to 5 rows unless the user explicitly asks otherwise.\n- If the tool returns 'Error:', revise the SQL and try again.\n- Prefer explicit column lists; avoid SELECT *.\n",
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'system',
   'name': None,
   'id': '8102d112-9856-44a7-b133-e4f97b186d6d'}},
 {'type': 'human',
  'data': {'content': 'Which table has the largest number of entries?',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': 'a8e79bde-19e9-44f0-81a2-4e25d00c9aaf'}},
 {'type': 'ai',
  'data': {'content': '',
   'additional_kwargs': {'

In [6]:
converted

[SystemMessage(content='You are helpful.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hello', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hi', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [7]:
from langchain_core.messages import HumanMessage
human_msg = HumanMessage("Hello, how are you?")

from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
agent = create_agent(
    model = "openai:gpt-5-nano",
    system_prompt= SystemMessage("You are a full-stack comedian")
)
result = agent.invoke({"messages":[human_msg]})

result

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='7e885a99-3aff-42ad-9c64-14fc217a8183'),
  AIMessage(content='Hey there! I’m doing great—my punchlines are compiling and my puns are fully optimized. How about you?\n\nWant a quick joke, a coding tip, or help with something serious (but still funny)? Here’s a teaser if you’re in the mood: Why do programmers prefer dark mode? Because light attracts bugs. Want another, or tell me what you need?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 662, 'prompt_tokens': 22, 'total_tokens': 684, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DicTR6YwIY0eiCSx6OHBAGzf6yQqx', 'servi

In [8]:
len(result["messages"])

2

In [9]:
result["messages"][-1]

AIMessage(content='Hey there! I’m doing great—my punchlines are compiling and my puns are fully optimized. How about you?\n\nWant a quick joke, a coding tip, or help with something serious (but still funny)? Here’s a teaser if you’re in the mood: Why do programmers prefer dark mode? Because light attracts bugs. Want another, or tell me what you need?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 662, 'prompt_tokens': 22, 'total_tokens': 684, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DicTR6YwIY0eiCSx6OHBAGzf6yQqx', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e540e-7fcd-7a53-953e-36c174c590da-0', tool_calls=[], invalid_tool_ca

In [10]:
result["messages"][-1].content

'Hey there! I’m doing great—my punchlines are compiling and my puns are fully optimized. How about you?\n\nWant a quick joke, a coding tip, or help with something serious (but still funny)? Here’s a teaser if you’re in the mood: Why do programmers prefer dark mode? Because light attracts bugs. Want another, or tell me what you need?'

In [11]:
type(result["messages"][0])

langchain_core.messages.human.HumanMessage

In [12]:
type(result["messages"][1])

langchain_core.messages.ai.AIMessage

In [13]:
type(result["messages"])

list

In [14]:
type(result)

dict

In [15]:
result.keys()

dict_keys(['messages'])

In [16]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Hey there! I’m doing great—my punchlines are compiling and my puns are fully optimized. How about you?

Want a quick joke, a coding tip, or help with something serious (but still funny)? Here’s a teaser if you’re in the mood: Why do programmers prefer dark mode? Because light attracts bugs. Want another, or tell me what you need?



In [17]:
for i,msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Hello, how are you?
================================== Ai Message ==================================

Hey there! I’m doing great—my punchlines are compiling and my puns are fully optimized. How about you?

Want a quick joke, a coding tip, or help with something serious (but still funny)? Here’s a teaser if you’re in the mood: Why do programmers prefer dark mode? Because light attracts bugs. Want another, or tell me what you need?


In [18]:
result = agent.invoke(
    {"messages":{"role":"user","content":"Write a haiku about sprinters"}}
)
result["messages"][-1].content

'Sprinters aim for dawn\nLanes blaze as runners fly high\nTime slips through the line'

In [19]:
from langchain_core.tools import tool

In [20]:
@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [21]:
agent = create_agent(
    model="openai:gpt-5",
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [22]:
result = agent.invoke({'messages':"please write me a poem"})

checking haiku, it has 3 lines:
 Whistle splits cool dawn
Footfalls drum the track like rain
Breath chases the tape


In [23]:
for i,msg in enumerate(result['messages']):
    msg.pretty_print()

================================ Human Message =================================

please write me a poem
================================== Ai Message ==================================
Tool Calls:
  check_haiku_lines (call_XOc2OBaBHWphavk1dP4ZNvZz)
 Call ID: call_XOc2OBaBHWphavk1dP4ZNvZz
  Args:
    text: Whistle splits cool dawn
Footfalls drum the track like rain
Breath chases the tape
================================= Tool Message =================================
Name: check_haiku_lines

Correct, this haiku has 3 lines.
================================== Ai Message ==================================

Whistle splits cool dawn
Footfalls drum the track like rain
Breath chases the tape


In [24]:
result['messages'][-1]

AIMessage(content='Whistle splits cool dawn\nFootfalls drum the track like rain\nBreath chases the tape', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 233, 'total_tokens': 256, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DicTsZyMyYhQsSsreJkHEtL3iQ6hV', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e540e-e94f-7b23-89d1-3cd257692470-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 233, 'output_tokens': 23, 'total_tokens': 256, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [25]:
result['messages'][-1].usage_metadata

{'input_tokens': 233,
 'output_tokens': 23,
 'total_tokens': 256,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 0}}

In [26]:
result['messages'][-1].response_metadata

{'token_usage': {'completion_tokens': 23,
  'prompt_tokens': 233,
  'total_tokens': 256,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-DicTsZyMyYhQsSsreJkHEtL3iQ6hV',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

## Streaming Options in Langchain

In [27]:
from dotenv import load_dotenv,dotenv_values

In [28]:
config['MODEL_NAME']

'openai:gpt-5-mini'

In [29]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
agent = create_agent(
    model= config['MODEL_NAME'],
    system_prompt = SystemMessage("You are a full-stack comedian")
)

In [30]:
result = agent.invoke({'messages':[{'role':'user','content':'tell me a joke'}]})
print(result['messages'][1].content)

Why do programmers prefer dark mode? Because light attracts bugs.

Why do programmers confuse Halloween and Christmas? Because Oct 31 == Dec 25.

I'm a full-stack comedian — I craft front-end laughs and handle back-end delivery. If the punchline doesn't render, try clearing your cache and refreshing your sense of humor.


In [31]:
# streaming mode : values 
for step in agent.stream(
    {'messages':[{'role':'user','content':'tell me a joke'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

tell me a joke
================================== Ai Message ==================================

I'm a full-stack comedian: my jokes have great UI, solid APIs, and terrible scalability.


In [32]:
# stream data token by token - lowest latency possible. 
for token,metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode='messages'
):
    print(f"{token.content}",end="")

We wake to giggles, pancakes flipping high,
Sunbeams paste paper smiles in the sky.
Socks go missing—one hides under the bed,
A dinosaur bookmark naps near your head.

Grandma hums a tune that makes the spoons all dance,
Dad tries to tango but trips on his pants.
Little hands build castles out of blankets and cheer,
While the dog insists the crown belongs to his ear.

Outside, the trees clap leaves for the breeze,
Clouds parade slowly like ships on soft seas.
Evening tucks the day in with a cozy, warm light,
Stars wink goodnight like fireflies in flight.

We share one last story, one last silly rhyme—
Close your eyes now, tomorrow waits for more time.

In [33]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
import pprint

def get_weather(city):
    """Get weather for a given city"""
    writer = get_stream_writer()
    writer(f"looking up data for city : {city}")
    writer(f"Acquired data for city : {city}")
    return f"It's always sunny in {city}"

agent = create_agent(
    model=config['MODEL_NAME'],
    tools = [get_weather]
)

for step in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"]
):
    print(pprint.pprint(step))
    
    print('------'*10)

('values',
 {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='8fa4672f-cd9c-4cf7-b0ce-b32929b2b0d3')]})
None
------------------------------------------------------------
('values',
 {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='8fa4672f-cd9c-4cf7-b0ce-b32929b2b0d3'),
               AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 90, 'prompt_tokens': 132, 'total_tokens': 222, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DicUBXTecgr5LkYxKtrxY1yAtMgM9', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}

## Exploring Tools

In [34]:
from typing import Literal
from langchain.tools import tool
from langgraph.config import get_stream_writer
from langchain.messages import SystemMessage,ToolMessage

In [35]:
@tool
def real_number_calculator(a, b, operation):
    """Perform basic arithmetic operations on two real numbers."""
    print(" -> Invoking calculator tool")
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [36]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage,ToolMessage
agent = create_agent(
    model=config['MODEL_NAME'],
    tools = [real_number_calculator],
    system_prompt = SystemMessage("You are a helpful assistant")
)

In [37]:
for step in agent.stream(
    {'messages':[{'role':'user','content':'what is 3.1125 * 4.1234'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

what is 3.1125 * 4.1234
================================== Ai Message ==================================
Tool Calls:
  real_number_calculator (call_CZKFpjRMaFaa0pSxG7kqzlHi)
 Call ID: call_CZKFpjRMaFaa0pSxG7kqzlHi
  Args:
    a: 3.1125
    b: 4.1234
    operation: multiply
 -> Invoking calculator tool
================================= Tool Message =================================
Name: real_number_calculator

12.8340825
================================== Ai Message ==================================

12.8340825


In [38]:
# since the inputs are integers : this may not invoke the tools 
for step in agent.stream(
    {'messages':[{'role':'user','content':'what is 3 * 4'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

what is 3 * 4
================================== Ai Message ==================================

3 × 4 = 12


In [39]:
# adding a more detailed description
from typing import Literal
from langchain.tools import tool
@tool(
    "calculator",
    parse_docstring=True,
    description=(
        "Perform basic arithmetic operations on two real numbers."
        "Use this whenever you have operations on any numbers, even if they are integers."
    ),
)
def real_number_calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic operations on two real numbers.

    Args:
        a (float): The first number.
        b (float): The second number.
        operation (Literal["add", "subtract", "multiply", "divide"]):
            The arithmetic operation to perform.

            - `"add"`: Returns the sum of `a` and `b`.
            - `"subtract"`: Returns the result of `a - b`.
            - `"multiply"`: Returns the product of `a` and `b`.
            - `"divide"`: Returns the result of `a / b`. Raises an error if `b` is zero.

    Returns:
        float: The numerical result of the specified operation.

    Raises:
        ValueError: If an invalid operation is provided or division by zero is attempted.
    """
    print("🧮  Invoking calculator tool")
    # Perform the specified operation
    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Invalid operation: {operation}")

In [40]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[real_number_calculator],
    system_prompt="You are a helpful assistant",
)

for step in agent.stream(
    {'messages':[{'role':'user','content':'what is 3 * 4'}]},
    stream_mode='values'
):
    step['messages'][-1].pretty_print()

================================ Human Message =================================

what is 3 * 4
================================== Ai Message ==================================
Tool Calls:
  calculator (call_eD094BnWCQykG6ewjV7niQQD)
 Call ID: call_eD094BnWCQykG6ewjV7niQQD
  Args:
    a: 3
    b: 4
    operation: multiply
🧮  Invoking calculator tool
================================= Tool Message =================================
Name: calculator

12.0
================================== Ai Message ==================================

12


# Tools with MCP

In [41]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio
nest_asyncio.apply()

mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} MCP tools: {[t.name for t in mcp_tools]}")

Loaded 5 MCP tools: ['add_time', 'compare_time', 'convert_timezone', 'current_time', 'relative_time']


In [42]:
from langchain.agents import create_agent

agent_with_mcp = create_agent(
    model="openai:gpt-5",
    tools=mcp_tools,
    system_prompt="You are a helpful assistant",
)

result = await agent_with_mcp.ainvoke(
    {"messages": [{"role": "user", "content": "What's the time in SF right now?"}]}
)
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the time in SF right now?
================================== Ai Message ==================================
Tool Calls:
  current_time (call_4QjS12Y7NamlwrBDCdz90pMD)
 Call ID: call_4QjS12Y7NamlwrBDCdz90pMD
  Args:
    format: Monday, January 2, 2006 3:04 PM MST
    timezone: America/Los_Angeles
================================= Tool Message =================================
Name: current_time

[{'type': 'text', 'text': 'Saturday, May 23, 2026 1:59 AM PDT', 'id': 'lc_275f2097-3cdd-4912-b51f-8f04d0132dd2'}]
================================== Ai Message ==================================

It’s currently Saturday, May 23, 2026, 1:59 AM PDT in San Francisco.


## Adding short term memory to the agent

In [43]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase


from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)


In [44]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_XchVFUsDwTaU2VKbyJ0glMyT)
 Call ID: call_XchVFUsDwTaU2VKbyJ0glMyT
  Args:
    query: SELECT name FROM sqlite_master WHERE type = 'table' AND lower(name) LIKE '%invoice%' ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Invoice',), ('InvoiceLine',)]
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_kqZrLtUNEOOlxtFf35dbJy9K)
 Call ID: call_kqZrLtUNEOOlxtFf35dbJy9K
  Args:
    query: SELECT sql FROM sqlite_master WHERE type='table' AND name='Invoice';
================================= Tool Message =================================
Name: execute_sql

[('CREATE TABLE [Invoice]\n(\n    [InvoiceId] INTEG

In [45]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?
================================== Ai Message ==================================

Could you clarify which titles you mean (e.g., book titles, movie titles, job titles)? If you can specify the table name and any filters (such as date range, category, author) and how many rows you want, I’ll query and list them.


In [46]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)


question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_MmnsVskMEIyErkZ8SiWtyE1O)
 Call ID: call_MmnsVskMEIyErkZ8SiWtyE1O
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type='table' AND lower(name) LIKE '%invoice%' LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Invoice', 'CREATE TABLE [Invoice]\n(\n    [InvoiceId] INTEGER  NOT NULL,\n    [CustomerId] INTEGER  NOT NULL,\n    [InvoiceDate] DATETIME  NOT NULL,\n    [BillingAddress] NVARCHAR(70),\n    [BillingCity] NVARCHAR(40),\n    [BillingState] NVARCHAR(40),\n    [BillingCountry] NVARCHAR(40),\n    [BillingPostalCode]...'), ('InvoiceLine', 'CREATE TABLE [InvoiceLine]\n(\n    [InvoiceLineId] INTEGER  NOT NULL,\n    [InvoiceId] INTEGER  NOT NULL,\n    [TrackId] IN

# Structured Outputs 

In [47]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str


agent = create_agent(model="openai:gpt-5-mini", response_format=ContactInfo)

recorded_conversation = """We talked with John Doe. He works over at Example. His number is, let's see, 
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [48]:
# multiple data types are supported
# pydantic BaseModel
# TypedDict
# dataclasses 
# json schema (dict) 

from langchain.agents import create_agent
from pydantic import BaseModel


class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str


agent = create_agent(model="openai:gpt-5-mini", response_format=ContactInfo)

recorded_conversation = """ We talked with John Doe. He works over at Example. His number is, let's see, 
five, five, five, one two three, four, five, six seven. Did you get that?
And, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch."""

result = agent.invoke(
    {"messages": [{"role": "user", "content": recorded_conversation}]}
)

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='555-123-4567')

ContactInfo(name='John Doe', email='john@example.com', phone='555-123-4567')

In [49]:
result

{'messages': [HumanMessage(content=" We talked with John Doe. He works over at Example. His number is, let's see, \nfive, five, five, one two three, four, five, six seven. Did you get that?\nAnd, his email was john at example.com. He wanted to order 50 boxes of Captain Crunch.", additional_kwargs={}, response_metadata={}, id='dad6bb24-3700-4339-b151-e1ba36083d72'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"555-123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 674, 'prompt_tokens': 214, 'total_tokens': 888, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DicXKec4p2Q4rMmRQPn5i0QdUOHZw', 'service_tier': 'default', 'fin

In [50]:
result.keys()

dict_keys(['messages', 'structured_response'])

# Dynamic Prompt

In [51]:
from langchain_community.utilities import SQLDatabase
from dataclasses import dataclass
db = SQLDatabase.from_uri("sqlite:///Chinook.db")

@dataclass
class RuntimeContext:
    is_employee: bool
    db: SQLDatabase
    
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

SYSTEM_PROMPT_TEMPLATE = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
{table_limits}
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""
        

In [52]:
from langchain.agents.middleware.types import ModelRequest, dynamic_prompt

@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    if not request.runtime.context.is_employee:
        table_limits = "- Limit access to these tables: Album, Artist, Genre, Playlist, PlaylistTrack, Track."
    else:
        table_limits = ""
    return SYSTEM_PROMPT_TEMPLATE.format(table_limits=table_limits)

from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    middleware=[dynamic_system_prompt],
    context_schema=RuntimeContext,
)

In [53]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=False, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================

I can’t answer that with the tables I’m allowed to query (Album, Artist, Genre, Playlist, PlaylistTrack, Track). To find “the most costly purchase by Frank Harris,” I need access to Customer, Invoice, and InvoiceLine.

Please confirm:
- Do you mean the highest total invoice (sum of all items on a single receipt) or the single most expensive item he bought?
- May I query the Customer, Invoice, and InvoiceLine tables?

If approved for “highest total invoice,” I would run a query like:
SELECT i.InvoiceId,
       i.InvoiceDate,
       ROUND(SUM(il.UnitPrice * il.Quantity), 2) AS InvoiceTotal
FROM Customer c
JOIN Invoice i ON i.CustomerId = c.CustomerId
JOIN InvoiceLine il ON il.InvoiceId = i.InvoiceId
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
GROUP BY i.InvoiceId, i.I

In [54]:
question = "What is the most costly purchase by Frank Harris?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(is_employee=True, db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the most costly purchase by Frank Harris?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_xFMd1Nbk6nyQpGHN8Jo60qia)
 Call ID: call_xFMd1Nbk6nyQpGHN8Jo60qia
  Args:
    query: SELECT name, sql FROM sqlite_master WHERE type='table' AND (lower(name) LIKE '%customer%' OR lower(name) LIKE '%client%' OR lower(name) LIKE '%user%' OR lower(name) LIKE '%person%' OR lower(name) LIKE '%people%') ORDER BY name LIMIT 5;
================================= Tool Message =================================
Name: execute_sql

[('Customer', 'CREATE TABLE [Customer]\n(\n    [CustomerId] INTEGER  NOT NULL,\n    [FirstName] NVARCHAR(40)  NOT NULL,\n    [LastName] NVARCHAR(20)  NOT NULL,\n    [Company] NVARCHAR(80),\n    [Address] NVARCHAR(70),\n    [City] NVARCHAR(40),\n    [State] NVARCHAR(40),\n    [Country] NVARCHAR(40),\n    [PostalCode]...')]
==========

# Human in the loop

In [55]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

from langchain_core.tools import tool
from langgraph.runtime import get_runtime


@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db
    
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
- If the database is offline, ask user to try again later without further comment.
"""


In [56]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5",
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    context_schema=RuntimeContext,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"execute_sql": {"allowed_decisions": ["approve", "reject"]}},
        ),
    ],
)

In [57]:
from langgraph.types import Command

question = "What are the names of all the employees?"
config = {"configurable": {"thread_id": "1"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db)
)

if "__interrupt__" in result:
    description = result['__interrupt__'][-1].value['action_requests'][-1]['description']
    print(description)
    result = agent.invoke(
        Command(
            resume={
                "decisions": [{"type": "reject", "message": "the database is offline."}]
            }
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )
print(result["messages"][-1].content)

Tool execution requires approval

Tool: execute_sql
Args: {'query': 'SELECT name FROM employees'}
The database is currently offline. Please try again later.


In [58]:
config = {"configurable": {"thread_id": "2"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=config,
    context=RuntimeContext(db=db)
)

while "__interrupt__" in result:
    description = result['__interrupt__'][-1].value['action_requests'][-1]['description']
    print(description)
    
    result = agent.invoke(
        Command(
            resume={"decisions": [{"type": "approve"}]}
        ),
        config=config,  # Same thread ID to resume the paused conversation
        context=RuntimeContext(db=db),
    )

for msg in result["messages"]:
    msg.pretty_print()

Tool execution requires approval

Tool: execute_sql
Args: {'query': "SELECT name FROM sqlite_master WHERE type = 'table' AND (\n  LOWER(name) LIKE '%employee%' OR\n  LOWER(name) LIKE '%staff%' OR\n  LOWER(name) LIKE '%person%' OR\n  LOWER(name) LIKE '%user%'\n) ORDER BY name LIMIT 5;"}
Tool execution requires approval

Tool: execute_sql
Args: {'query': "SELECT sql FROM sqlite_master WHERE type='table' AND name='Employee';"}
Tool execution requires approval

Tool: execute_sql
Args: {'query': "SELECT FirstName || ' ' || LastName AS FullName FROM Employee ORDER BY LastName, FirstName;"}
================================ Human Message =================================

What are the names of all the employees?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_kTAACSw2Cd1KcR5ZX4NV6vsl)
 Call ID: call_kTAACSw2Cd1KcR5ZX4NV6vsl
  Args:
    query: SELECT name FROM sqlite_master WHERE type = 'table' AND (
  LOWER(name) LIKE '%employee%